# Feature Quality Analysis

Deep analysis of each feature: SHAP, drop-column importance, permutation importance, null importance, and correlation.


## 1. Setup


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score
)
from sklearn.inspection import permutation_importance
from lightgbm import LGBMClassifier
import shap
print("Imports done")


## 2. Load Data & Feature Engineering

Use the full set (all candidate features) for analysis.


In [ ]:
def engineer_features(X):
    X = X.copy()
    X["Amount_log"] = np.log1p(X["Amount"])
    X["Hour"] = (X["Time"] // 3600) % 24
    X["Hour_sin"] = np.sin(2 * np.pi * X["Hour"] / 24)
    X["Hour_cos"] = np.cos(2 * np.pi * X["Hour"] / 24)
    v_cols = [c for c in X.columns if c.startswith("V")]
    X["V_mean"] = X[v_cols].mean(axis=1)
    X["V_std"] = X[v_cols].std(axis=1)
    X["V_max"] = X[v_cols].max(axis=1)
    X["V_min"] = X[v_cols].min(axis=1)
    X["V_outliers_count"] = (np.abs(X[v_cols]) > 3).sum(axis=1)
    X["V_range"] = X["V_max"] - X["V_min"]
    X["V_skew"] = X[v_cols].skew(axis=1)
    X["V_mad"] = X[v_cols].sub(X[v_cols].median(axis=1), axis=0).abs().median(axis=1)
    X["V_sum_sq"] = (X[v_cols]**2).sum(axis=1)
    X["Amount_rank"] = X["Amount_log"].rank(pct=True)
    X["Time_since_prev"] = X["Time"].diff().fillna(0)
    return X

df = pd.read_csv("data/raw/creditcard.csv")
TARGET = "Class"
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
ratio = (y_train == 0).sum() / (y_train == 1).sum()

X_train = engineer_features(X_train)
X_test = engineer_features(X_test)
DROP = ["Time", "Amount", "Hour"]
X_train.drop(columns=DROP, inplace=True)
X_test.drop(columns=DROP, inplace=True)

SCALE_COLS = ["Amount_log", "Hour_sin", "Hour_cos", "V_mean", "V_std",
              "V_max", "V_min", "V_outliers_count", "V_range", "V_skew",
              "V_mad", "V_sum_sq", "Amount_rank", "Time_since_prev"]

scaler = StandardScaler()
X_train[SCALE_COLS] = scaler.fit_transform(X_train[SCALE_COLS])
X_test[SCALE_COLS] = scaler.transform(X_test[SCALE_COLS])

v_cols = [c for c in X_train.columns if c.startswith("V")]
eng_cols = [c for c in X_train.columns if not c.startswith("V")]
print(f"V-features: {len(v_cols)}  Engineered features: {len(eng_cols)}")
print(f"Total: {X_train.shape[1]}")


## 3. Train Baseline LightGBM


In [ ]:
model = LGBMClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=8,
    num_leaves=32, class_weight="balanced",
    random_state=42, n_jobs=-1, verbose=-1
)
model.fit(X_train, y_train)
probs = model.predict_proba(X_test)[:, 1]
preds = (probs >= 0.5).astype(int)

baseline_pr = average_precision_score(y_test, probs)
baseline_roc = roc_auc_score(y_test, probs)
baseline_f1 = f1_score(y_test, preds)

print(f"Baseline LightGBM (all features)")
print(f"  PR-AUC : {baseline_pr:.4f}")
print(f"  ROC-AUC: {baseline_roc:.4f}")
print(f"  F1     : {baseline_f1:.4f}")

fi = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 10))
fi.plot.barh(ax=ax)
ax.set_xlabel("Importance")
ax.set_title("LightGBM Built-in Feature Importance")
plt.tight_layout()
plt.show()


## 4. SHAP Analysis

Shows whether each feature pushes predictions toward fraud or non-fraud. Uses a random subset of test data (2000 rows) for speed.


In [ ]:
X_sample = X_test.sample(2000, random_state=42)
y_sample = y_test.loc[X_sample.index]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

# Summary beeswarm plot
shap.summary_plot(shap_values, X_sample, show=False)
plt.tight_layout()
plt.show()

# Bar plot of mean absolute SHAP values
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
plt.tight_layout()
plt.show()


## 5. Drop-Column Importance

Remove each feature, retrain, measure PR-AUC change. **Negative delta means the feature hurts the model.**


In [ ]:
drop_results = []
for col in X_train.columns:
    Xtr = X_train.drop(columns=[col])
    Xte = X_test.drop(columns=[col])
    m = LGBMClassifier(n_estimators=200, learning_rate=0.1, max_depth=8,
                       num_leaves=32, class_weight="balanced",
                       random_state=42, n_jobs=-1, verbose=-1)
    m.fit(Xtr, y_train)
    p = m.predict_proba(Xte)[:, 1]
    pr = average_precision_score(y_test, p)
    drop_results.append({"Feature": col, "PR_AUC": pr, "Delta": pr - baseline_pr})

drop_df = pd.DataFrame(drop_results).sort_values("Delta", ascending=False)

fig, ax = plt.subplots(figsize=(10, 10))
colors = ["green" if d > 0 else "red" for d in drop_df["Delta"].values]
drop_df.set_index("Feature")["Delta"].plot.barh(ax=ax, color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("PR-AUC change when feature removed (delta)")
ax.set_title("Drop-Column Importance (negative = feature hurts)")
plt.tight_layout()
plt.show()

print("Features that HURT the model (PR-AUC increased when removed):")
hurting = drop_df[drop_df["Delta"] > 0]
if len(hurting) > 0:
    for _, r in hurting.iterrows():
        print(f"  {r['Feature']:25s}: +{r['Delta']:.4f} PR-AUC")
else:
    print("  None")

print("\nFeatures that HELP the model most (PR-AUC dropped when removed):")
helping = drop_df[drop_df["Delta"] < 0].head(10)
for _, r in helping.iterrows():
    print(f"  {r['Feature']:25s}: {r['Delta']:.4f} PR-AUC")


## 6. Permutation Importance

Shuffle each feature, measure PR-AUC drop. Model-agnostic.


In [ ]:
perm = permutation_importance(
    model, X_test, y_test, n_repeats=5, random_state=42,
    scoring="average_precision"
)
perm_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": perm.importances_mean,
    "Std": perm.importances_std,
}).sort_values("Importance", ascending=False)

fig, ax = plt.subplots(figsize=(10, 10))
colors = ["green" if v > 0 else "red" for v in perm_df["Importance"].values]
perm_df.set_index("Feature")["Importance"].plot.barh(ax=ax, color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("PR-AUC drop when shuffled")
ax.set_title("Permutation Importance")
plt.tight_layout()
plt.show()

print("Features with zero or negative permutation importance:")
noise = perm_df[perm_df["Importance"] <= 0]
for _, r in noise.iterrows():
    print(f"  {r['Feature']:25s}: {r['Importance']:.4f}")


## 7. Null Importance

Compare real feature importances against a null distribution (5x shuffled target). Features whose real importance doesn't exceed noise are candidates for removal.


In [ ]:
np.random.seed(42)
null_importances = {col: [] for col in X_train.columns}

for i in range(5):
    y_shuffled = y_train.sample(frac=1, random_state=i).values
    m = LGBMClassifier(n_estimators=200, learning_rate=0.1, max_depth=8,
                       num_leaves=32, class_weight="balanced",
                       random_state=42, n_jobs=-1, verbose=-1)
    m.fit(X_train, y_shuffled)
    for col, imp in zip(X_train.columns, m.feature_importances_):
        null_importances[col].append(imp)

null_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Real_Importance": model.feature_importances_,
    "Null_Mean": [np.mean(null_importances[col]) for col in X_train.columns],
    "Null_Max": [np.max(null_importances[col]) for col in X_train.columns],
})
null_df["Signal_Noise_Ratio"] = null_df["Real_Importance"] / (null_df["Null_Mean"] + 1)
null_df["Is_Noise"] = null_df["Real_Importance"] <= null_df["Null_Max"]

print("Features that look like noise (real importance <= max null importance):")
noise_feats = null_df[null_df["Is_Noise"]].sort_values("Real_Importance")
for _, r in noise_feats.iterrows():
    print(f"  {r['Feature']:25s}: real={r['Real_Importance']:.0f}  null_max={r['Null_Max']:.0f}")

print("\nTop features by signal-to-noise ratio:")
signal = null_df.sort_values("Signal_Noise_Ratio", ascending=False).head(15)
for _, r in signal.iterrows():
    print(f"  {r['Feature']:25s}: SNR={r['Signal_Noise_Ratio']:.1f}")


## 8. Correlation Analysis & Multicollinearity


In [ ]:
corr = X_train.corr()

# Correlation between engineered features and V-features
fig, ax = plt.subplots(figsize=(14, 6))
corr_subset = corr.loc[eng_cols, v_cols]
im = ax.imshow(corr_subset.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(v_cols)))
ax.set_xticklabels([f"V{i+1}" for i in range(len(v_cols))], fontsize=6)
ax.set_yticks(range(len(eng_cols)))
ax.set_yticklabels(eng_cols)
plt.colorbar(im, ax=ax, shrink=0.6)
ax.set_title("Correlation: Engineered features vs V-features")
plt.tight_layout()
plt.show()

# Pairwise correlations among engineered features
eng_corr = corr.loc[eng_cols, eng_cols]
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(eng_corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(eng_cols)))
ax.set_xticklabels(eng_cols, rotation=45, ha="right")
ax.set_yticks(range(len(eng_cols)))
ax.set_yticklabels(eng_cols)
for i in range(len(eng_cols)):
    for j in range(len(eng_cols)):
        val = eng_corr.values[i, j]
        if abs(val) > 0.5:
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, ax=ax, shrink=0.6)
ax.set_title("Correlation Among Engineered Features")
plt.tight_layout()
plt.show()

# High correlation pairs among engineered features
print("High-correlation pairs among engineered features (|r| > 0.7):")
for i in range(len(eng_cols)):
    for j in range(i+1, len(eng_cols)):
        val = eng_corr.values[i, j]
        if abs(val) > 0.7:
            print(f"  {eng_cols[i]:25s} vs {eng_cols[j]:25s}: r={val:.3f}")


## 9. Build Refined Feature Set

Drop features that fail multiple tests (hurt in drop-column, zero/null in permutation, noise in null importance, or redundant through correlation).


In [ ]:
# Collect evidence
hurting_features = set(drop_df[drop_df["Delta"] > 0]["Feature"])
noise_permutation = set(perm_df[perm_df["Importance"] <= 0]["Feature"])
noise_null = set(null_df[null_df["Is_Noise"]]["Feature"])

# Redundant: engineered features with |r| > 0.7 to V-features AND low permutation importance
redundant = set()
for col in eng_cols:
    max_corr = max(abs(corr.loc[col, v_cols]))
    perm_val = perm_df[perm_df["Feature"] == col]["Importance"].values[0]
    if max_corr > 0.7 and perm_val < 0.001:
        redundant.add(col)

all_suspect = hurting_features | noise_permutation | noise_null | redundant
refined_cols = [c for c in X_train.columns if c not in all_suspect]

print("Features flagged for removal:")
print(f"  Hurting (drop-column Δ>0)  : {len(hurting_features)} -> {sorted(hurting_features)}")
print(f"  Zero/negative permutation   : {len(noise_permutation)} -> {sorted(noise_permutation)}")
print(f"  Noise (null importance)     : {len(noise_null)} -> {sorted(noise_null)}")
print(f"  Redundant (high corr+low)   : {len(redundant)} -> {sorted(redundant)}")
print(f"\nFeatures to KEEP ({len(refined_cols)}): {refined_cols}")
print(f"Features to DROP ({len(X_train.columns) - len(refined_cols)}): {sorted(all_suspect)}")


## 10. Final Validation

Train model on refined set vs original full set.


In [ ]:
Xtr_ref = X_train[refined_cols]
Xte_ref = X_test[refined_cols]

model_ref = LGBMClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=8,
    num_leaves=32, class_weight="balanced",
    random_state=42, n_jobs=-1, verbose=-1
)
model_ref.fit(Xtr_ref, y_train)
probs_ref = model_ref.predict_proba(Xte_ref)[:, 1]
preds_ref = (probs_ref >= 0.5).astype(int)

ref_pr = average_precision_score(y_test, probs_ref)
ref_roc = roc_auc_score(y_test, probs_ref)
ref_f1 = f1_score(y_test, preds_ref)

print(f"{'Metric':20s} {'Full Set':>10s} {'Refined':>10s} {'Delta':>10s}")
print("-" * 50)
print(f"{'PR-AUC':20s} {baseline_pr:>10.4f} {ref_pr:>10.4f} {ref_pr - baseline_pr:>+10.4f}")
print(f"{'ROC-AUC':20s} {baseline_roc:>10.4f} {ref_roc:>10.4f} {ref_roc - baseline_roc:>+10.4f}")
print(f"{'F1':20s} {baseline_f1:>10.4f} {ref_f1:>10.4f} {ref_f1 - baseline_f1:>+10.4f}")

# Per-feature verdict table
verdict = []
for col in X_train.columns:
    tests_failed = []
    if col in hurting_features:
        tests_failed.append("DropCol+")
    if col in noise_permutation:
        tests_failed.append("Perm0")
    if col in noise_null:
        tests_failed.append("NullNoise")
    if col in redundant:
        tests_failed.append("Redundant")
    if not tests_failed:
        tests_failed.append("PASS")

    fi_val = fi.get(col, 0)
    verdict.append({
        "Feature": col,
        "Importance": fi_val,
        "Verdict": "KEEP" if col in refined_cols else "DROP",
        "Reason": ", ".join(tests_failed),
    })

verdict_df = pd.DataFrame(verdict).sort_values("Importance", ascending=False)
print("\n\nPer-Feature Verdict:")
print(verdict_df.to_string(index=False))
